# EJEMPLO 1: PRUEBA Z DE UNA MUESTRA (UNILATERAL)

Supongamos que trabajamos en una empresa de streaming (tipo Netflix o Spotify) y que estamos probando un nuevo sistema de recomendación. Y sabemos que el sistema actual tiene un desempeño promedio (que mide la calidad de las recomendaciones) de 0.75 y una desviación de 0.05, un valor que se ha obtenido con muchas pruebas realizadas con muchísimos sets de datos

**Queremos saber si el nuevo sistema de recomendación es mejor que el usado actualmente**

## 1. Breve repaso: los pasos para llevar a cabo una prueba de hipótesis

Se sugiere revisar la [lección 12 del curso Estadística Inferencial: Fundamentos](https://codificandobits.com/curso/estadistica-inferencial-fundamentos/12-pasos-para-aplicar-pruebas-de-hipotesis/).

A continuación se resumen estos pasos:

1. Definir el problema del negocio
2. Redactar el problema del negocio como un problema de Ciencia de Datos/Machine Learning
3. Definir $H_0$ y $H_1$
4. Definir $\alpha$
5. Definir $1-\beta$ (potencia de la prueba) y $n$ (tamaño de la muestra)
6. Recolectar y preparar los datos
7. Aplicar la prueba estadística para obtener el valor p
8. Aceptar o rechazar $H_0$ con base en (3), (4) y (7)
9. Evaluar el tamaño del efecto y la potencia posteriores a la prueba de hipótesis

Veamos cada uno de estos pasos para el problema que queremos resolver.



## 2. Desarrollo de la prueba de hipótesis (z-test de una muestra unilateral)

### 2.1. Paso 1: definir el problema del negocio

> **Queremos saber si el nuevo sistema de recomendación es mejor que el usado actualmente**

### 2.2. Paso 2: redactar el problema del negocio como un problema de Ciencia de Datos/Machine Learning

> ¿El nuevo sistema genera mejoras **estadísticamente significativas** en el desempeño?

### 2.3. Paso 3: definir $H_0$ y $H_1$

- $H_0$: el promedio de desempeño del nuevo sistema es igual al del modelo anterior $\rightarrow \bar{x} = \mu$
- $H_1$: el promedio de desempeño del nuevo modelo es **mayor** que el del modelo anterior $\rightarrow \bar{x} > \mu$ 

*Nota: recordemos que $\mu = 0.75$ y $\sigma = 0.05$*

### 2.4. Paso 4: definir $\alpha$

Asumiremos un nivel de significancia $\alpha = 0.05$. Podemos dibujar este nivel de significancia en la distribución Z usando alguna [herramienta online](https://www.infrrr.com/distributions/normal-distributions):

![](distribucion_z_test_unilateral.png)

En este gráfico:
- El eje horizontal es la variable $z$
- El valor de $z$ correspondiente a un $\alpha = 0.05$ es 1.645
- La región azul corresponde a la **cola derecha** (prueba unilateral porque queremos saber qué tan probable es que la diferencia esté en ese último 5%) y contiene el 5% de la distribución
- Si tras realizar la prueba el valor de $z$ calculado está dentro de la zona azul, nos inclinaremos por $H_1$ y rechazaremos $H_0$. De lo contrario nos inclinamos por $H_0$ y rechazamos $H_1$

### 2.5. Definir la potencia de la prueba ($1-\beta$) y el tamaño de la muestra ($n$)

Se sugiere revisar las lecciones 10 y 11 del curso Estadística Inferencial: Fundamentos:

- [Lección 11: errores tipo I y tipo II](https://codificandobits.com/curso/estadistica-inferencial-fundamentos/10-errores-tipo-i-y-tipo-ii/)
- [Lección 12: potencia de una prueba](https://codificandobits.com/curso/estadistica-inferencial-fundamentos/11-potencia-de-una-prueba/)

Para definir la potencia de una prueba debemos primero establecer el tamaño del efecto deseado:

<p></p>
<div style="background-color: #F7CAC9; color: black; padding: 10px;">
    El tamaño del efecto en este caso nos indica qué tan grande es el incremento en el desempeño que queremos detectar
</div>

Supongamos que un incremento de al menos 0.04 (de 0.75 a 0.79) es bastante bueno. En el caso de la prueba z el tamaño del efecto será:

$$d = \frac{\bar{x}-\mu}{\sigma}=\frac{0.04}{0.05}=0.8$$

Con este tamaño del efecto podemos calcular la potencia de la prueba ($1-\beta$):

<p></p>
<div style="background-color: #F7CAC9; color: black; padding: 10px;">
La potencia de una prueba es la probabilidad (1-𝜷) de rechazar correctamente la hipótesis nula cuando esta es falsa
</div>

Supongamos que queremos una potencia del 80%: es decir que si rechazamos la hipótesis nula tendremos un 80% de probabilidades de rechazarla cuando realmente es falsa:

$$1 - \beta = 0.8$$

Con todo lo anterior podemos calcular el tamaño de la muestra ($n$) que debemos recolectar para tener la potencia de la prueba, el tamaño del efecto y el nivel de significancia esperados. Para el caso de la prueba z el tamaño de la muestra se define con esta ecuación:

$$\begin{align*}

\\
n &= \left( \frac{z_\alpha + z_\beta}{d} \right)^2
\end{align*}
$$

donde:

- $z_\alpha$ es el valor de z asociado a 𝜶
- $z_\beta$ es el valor de z asociado a 1-𝛽

Tras hacer este cálculo obtenemos:

$$n = 9.6 \approx 10$$

En realidad este cálculo no tenemos que hacerlo de forma manual y en su lugar podemos usar la librerías "statsmodels" de Python:

In [27]:
!pip install statsmodels


[notice] A new release of pip available: 22.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [28]:
from statsmodels.stats.power import NormalIndPower

# Definir parámetros de nuestra prueba para el cálculo de n
diff = 0.04 # Diferencia esperada entre las medias
sigma = 0.05 # Desviación estándar en la población
effect_size = diff/sigma # Tamaño del efecto (0.8)
power = 0.8 # POtencia esperada de la prueba
alpha = 0.05 # Nivel de significancia

# Instancia de NormalIndPower
analisis = NormalIndPower()

# Y cálculo del tamaño de la muestra
n = analisis.solve_power(
    effect_size=effect_size,
    alpha = alpha,
    power=power,
    alternative='larger', # larger = unilateral derecho, smaller = unilateral izquierdo, two-sided: bilateral
    ratio = 0 # No estamos calculando proporción entre medias
)
print(f"Tamaño sugerido de la muestra: {n}")

Tamaño sugerido de la muestra: 9.66024513323637


Es decir que:

> **La muestra debe ser de al menos $n=10$ para lograr una potencia de 0.8, un tamaño del lefecto de 0.8 y un nivel de significancia de 0.05**

### 2.5. Paso 5: recolectar y preparar los datos

Supondremos que hemos recolectado $n=12$ sets de datos con los cuales probaremos el nuevo sistema de recomendación. A continuación podemos ver los desempeños del nuevo sistema obtenidos para este set de datos:

In [29]:
import pandas as pd

df = pd.read_csv('dataset_ztest_unilateral.csv')
df

,metrica
0,0.799144
1,0.819973
2,0.812830
3,0.794937
4,0.804214
5,0.826514
6,0.785733
7,0.805711
8,0.822659
9,0.801333


Y podemos verificar que tenemos 12 mediciones (12 datos). Veamos el promedio de estos desempeños:

In [30]:
df.mean()

metrica    0.807109
dtype: float64

El desempeño promedio de este nuevo sistema es de $\bar{x}=0.8$ que es mayor que el desempeño del sistema original ($\mu = 0.75$).

Pero recordemos que lo que nos interesa es **determinar si este incremento es estadísticamente significativo** o debido al azar.

Aprovechemos para verificar la normalidad de esta distribución:

In [31]:
from scipy.stats import shapiro

W, p_shapiro = shapiro(df['metrica'])
p_shapiro

0.9535694858018976

Como p>0.05 no rechazamos la hipótesis nula y por tanto los tienen una distribución normal.

### 2.6. Paso 6: aplicar la prueba estadística para obtener el valor p

Hagamos el análisis manual y luego veremos cómo hacerlo usando "statsmodels".

Comencemos calculando $z$:

$$z = \frac{\bar{x}-\mu}{\sigma/\sqrt{n}}=\frac{0.807-0.75}{0.05/\sqrt{12}}=3.94$$

A continuación verificamos dónde se encuentra ubicado este valor de $z$ dentro de la distribución:

![](distribucion_z_test_unilateral.png)

Y con esto verificamos que $z$ está en la zona sombreada y esto nos indica que podemos rechazar $H_0$ e inclinarnos por $H_1$.

Veamos como llegar a este mismo resultado usando "statsmodels":


In [32]:
from scipy.stats import norm
import numpy as np

# Parámetros de la población y de la muestra
mu = 0.75
sigma = 0.05
n = len(df)
x_barra = np.mean(df)

# Calcular z
z = (x_barra-mu)/(sigma/np.sqrt(n))

# Calcular p = 1 - (P(Z<=z))
p = 1 - norm.cdf(z) # Para unilateral izquierda el cálculo sería norm.cdf(z) (porque sería simplemente P(Z<=z))

print(f'z: {z}')
print(f'p: {p}')

z: 3.956653204207625
p: 3.8003574485578895e-05


### 2.7. Paso 7: aceptar o rechazar $H_0$

Y vemos que $p = 3.8x10^{-5} < 0.05$ y por tanto rechazamos la hipótesis nula (medias iguales) y nos inclinamos por $H_1$ (medias diferentes).

Y por tanto:

> El nuevo sistema de recomendación genera un incremento estadísticamente significativo en el desempeño (0.8 vs. 0.75)

### 2.8. Paso 8: evaluar el tamaño del efecto actualizado

Hemos visto que hay un efecto pero ¿qué tan grande es?

Simplemente recalculamos el tamaño del efecto con la media de los datos obtenidos:

In [33]:
# Tamaño del efecto actualizado
d = (np.mean(df)-mu)/sigma
print(f'Tamaño del efecto (d) actualizado: {d}')

Tamaño del efecto (d) actualizado: 1.1421873962696338


Esto quiere decir que:

> El nuevo desempeño se aleja una desviación estándar del valor original ✅✅✅

Además, teniendo en cuenta que no usamos el mínimo de datos (9.6), sino un poco más (12), verifiquemos la potencia final de esta prueba:

In [34]:
# Definir parámetros de la prueba actualizada (con el nuevo n)
n = len(df)
alpha = 0.05 # Nivel de significancia

# Instancia de NormalIndPower
analisis = NormalIndPower()

# Potencia actualizada
potencia = analisis.power(
    effect_size=d, # El valor calculado en el bloque de código anterior
    nobs1 = n, # tamaño de la muestra usada
    alpha = alpha,
    ratio = 0, # porque es prueba de 1 muestra
    alternative = 'larger' # porque es unilateral derecha
)
print(f'Potencia actualizada de la prueba: {potencia}')

Potencia actualizada de la prueba: 0.989605634631917


Lo anterior quiere decir que:

> Si el efecto que queríamos detectar (incremento en el desempeño) realmente existe, había un 98.9% de probabilidad de que esta prueba lo detectara ✅✅✅

**¡¡¡Es decir que con esto podemos estar bastante confiados de que el nuevo sistema efectivamente tiene un mejor desempeño que el anterior y que esto no se debe al azar!!!**